# Traffic Signal RL - SUMO Training (Google Colab)

Train a PPO agent to control traffic signals using SUMO scenarios.

**Runtime:** No GPU needed (CPU is fine for this).

### Mathematical Framework

We model traffic signal control as a **Markov Decision Process (MDP)** defined by the tuple:

$$\mathcal{M} = \langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$$

where:
- $\mathcal{S}$ — State space (queue lengths, waiting times, signal phase)
- $\mathcal{A}$ — Action space (discrete set of signal phase selections)
- $P(s' | s, a)$ — Transition probability (governed by SUMO traffic simulation)
- $R(s, a)$ — Reward function (penalizes waiting time and queue buildup)
- $\gamma \in [0, 1]$ — Discount factor (how much the agent values future vs. immediate rewards)

The goal is to learn a policy $\pi_\theta(a|s)$ that maximizes the expected discounted return:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T} \gamma^t R(s_t, a_t) \right]$$

## 1. Clone Repository & Install Dependencies

Before training, we need:
- **SUMO** (Simulation of Urban MObility) — provides the transition dynamics $P(s'|s,a)$ via microscopic traffic simulation
- **sumo-rl** — wraps SUMO as a Gymnasium environment, defining $\mathcal{S}$, $\mathcal{A}$, and $R$
- **stable-baselines3** — implements the PPO optimization algorithm for learning $\pi_\theta$
- **PyTorch** — neural network backend for parameterizing the policy $\pi_\theta(a|s)$ and value function $V_\phi(s)$

In [ ]:
# Clone the repository
!git clone https://github.com/madch3m/reinforcement-learning-grad.git
%cd reinforcement-learning-grad

# Install all dependencies
!pip install eclipse-sumo traci sumolib sumo-rl stable-baselines3 torch tensorboard -q
!pip install -r traffic_rl_project/requirements.txt -q

In [ ]:
import os
import sys

# Set SUMO_HOME
import sumo
os.environ["SUMO_HOME"] = os.path.dirname(sumo.__file__)

# Add project modules to path
sys.path.insert(0, os.path.join(os.getcwd(), "traffic_rl_project"))
sys.path.insert(0, os.path.join(os.getcwd(), "gradio_app"))

print(f"SUMO_HOME set to: {os.environ['SUMO_HOME']}")
print(f"Project root: {os.getcwd()}")
print(f"Python path includes: traffic_rl_project, gradio_app")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sumo_rl
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback

NETS_DIR = os.path.join(os.path.dirname(sumo_rl.__file__), "nets")
print("All imports OK")
print(f"Available scenarios: {os.listdir(NETS_DIR)}")

## 2. Create SUMO Environment

The environment defines the MDP components:

**State space** $\mathcal{S} \subset \mathbb{R}^n$ — a normalized observation vector including:
- Queue density per lane: $d_i = \frac{\text{vehicles in lane } i}{\text{lane capacity}}$, $d_i \in [0, 1]$
- Current signal phase (one-hot encoded)

**Action space** $\mathcal{A} = \{0, 1, \ldots, k-1\}$ — discrete phase selection, where $k$ is the number of valid signal phases.

**Reward function** — based on change in cumulative waiting time:

$$R(s_t, a_t) = -\sum_{v \in \text{vehicles}} \Delta w_v$$

where $\Delta w_v$ is the change in waiting time for vehicle $v$. This penalizes actions that increase total delay.

**Transition dynamics** $P(s'|s,a)$ — governed by SUMO's car-following (Krauss model) and lane-changing models. The simulation advances by `delta_time` seconds per step.

Choose a scenario by changing `SCENARIO` below:
- `"single-intersection"` — 4-way intersection (simplest, $|\mathcal{A}| = 2$)
- `"2way-single-intersection"` — 2-way intersection
- `"RESCO/cologne1"` — Real-world Cologne intersection
- `"RESCO/ingolstadt1"` — Real-world Ingolstadt intersection
- `"RESCO/grid4x4"` — 4x4 grid (advanced, multi-agent)

In [ ]:
# === CHANGE SCENARIO HERE ===
SCENARIO = "single-intersection"  # try "RESCO/cologne1" for real-world data
TOTAL_TIMESTEPS = 50_000          # increase to 200k+ for better results
# ============================

# Find the net and route files
scenario_dir = os.path.join(NETS_DIR, SCENARIO)
net_file = [f for f in os.listdir(scenario_dir) if f.endswith(".net.xml")][0]
rou_file = [f for f in os.listdir(scenario_dir) if f.endswith(".rou.xml")][0]

print(f"Scenario: {SCENARIO}")
print(f"Net file: {net_file}")
print(f"Route file: {rou_file}")

In [ ]:
def make_env():
    return sumo_rl.SumoEnvironment(
        net_file=os.path.join(scenario_dir, net_file),
        route_file=os.path.join(scenario_dir, rou_file),
        use_gui=False,
        num_seconds=5000,
        delta_time=5,
        yellow_time=3,
        min_green=10,
        max_green=50,
        single_agent=True,
    )

# Quick test
env = make_env()
obs, info = env.reset()
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Initial obs: {obs}")
env.close()
print("Environment test PASSED")

## 3. Train PPO Agent

**Proximal Policy Optimization (PPO)** optimizes the policy by maximizing a clipped surrogate objective:

$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$

where:
- $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ — probability ratio between new and old policy
- $\hat{A}_t$ — estimated advantage: $\hat{A}_t = R_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$ (how much better action $a_t$ is than average)
- $\epsilon = 0.2$ (default) — clipping range that prevents destructively large policy updates

**Hyperparameters explained:**
| Parameter | Symbol | Value | Meaning |
|---|---|---|---|
| `learning_rate` | $\alpha$ | $3 \times 10^{-4}$ | Step size for gradient descent on $L^{CLIP}$ |
| `gamma` | $\gamma$ | $0.99$ | Discount factor: $\gamma^t$ weights reward $t$ steps ahead |
| `n_steps` | $T$ | $2048$ | Steps collected per rollout before each policy update |
| `batch_size` | $B$ | $64$ | Mini-batch size for SGD within each epoch |
| `n_epochs` | $K$ | $10$ | Number of passes over collected data per update |
| `ent_coef` | $c_H$ | $0.01$ | Entropy bonus weight: $c_H \cdot H(\pi_\theta)$ encourages exploration |

The full loss combines policy, value, and entropy:

$$L(\theta, \phi) = L^{CLIP}(\theta) - c_V \cdot \underbrace{(V_\phi(s_t) - R_t)^2}_{\text{value loss}} + c_H \cdot \underbrace{H(\pi_\theta(\cdot|s_t))}_{\text{entropy bonus}}$$

In [ ]:
train_env = make_env()

model = PPO(
    "MlpPolicy",
    train_env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    ent_coef=0.01,
)

print(f"Training for {TOTAL_TIMESTEPS:,} timesteps...")
model.learn(total_timesteps=TOTAL_TIMESTEPS)
print("Training complete!")

train_env.close()

## 4. Evaluate Trained Agent vs Random Baseline

We evaluate the learned policy $\pi_\theta$ against a **random baseline** $\pi_{\text{rand}}$ where:

$$\pi_{\text{rand}}(a|s) = \frac{1}{|\mathcal{A}|} \quad \forall a \in \mathcal{A}$$

For each policy, we compute the **empirical expected return** over $N$ episodes:

$$\hat{J}(\pi) = \frac{1}{N} \sum_{i=1}^{N} G_i \quad \text{where} \quad G_i = \sum_{t=0}^{T_i} R(s_t^{(i)}, a_t^{(i)})$$

The **improvement** is measured as relative gain:

$$\text{Improvement} = \frac{\hat{J}(\pi_\theta) - \hat{J}(\pi_{\text{rand}})}{|\hat{J}(\pi_{\text{rand}})|} \times 100\%$$

During evaluation, the trained agent uses a **deterministic policy** $a = \arg\max_a \pi_\theta(a|s)$ (no exploration noise).

In [ ]:
def evaluate(model, env, n_episodes=5, name="Agent"):
    """Evaluate a model and return episode rewards + per-step trajectory for episode 1."""
    episode_rewards = []
    first_ep_step_rewards = []
    for ep in range(n_episodes):
        obs, info = env.reset()
        total_reward = 0.0
        step_rewards = []
        done = False
        while not done:
            if model is None:  # random baseline
                action = env.action_space.sample()
            else:
                action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            step_rewards.append(reward)
            done = terminated or truncated
        episode_rewards.append(total_reward)
        if ep == 0:
            first_ep_step_rewards = step_rewards
        print(f"  {name} Episode {ep+1}: reward = {total_reward:.2f}")
    return episode_rewards, first_ep_step_rewards


eval_env = make_env()
print("=== Trained PPO Agent ===")
ppo_rewards, ppo_step_rewards = evaluate(model, eval_env, n_episodes=3, name="PPO")
eval_env.close()

eval_env = make_env()
print("\n=== Random Baseline ===")
random_rewards, random_step_rewards = evaluate(None, eval_env, n_episodes=3, name="Random")
eval_env.close()

ppo_mean = float(np.mean(ppo_rewards))
random_mean = float(np.mean(random_rewards))
improvement = (ppo_mean - random_mean) / abs(random_mean) * 100 if random_mean != 0 else 0.0

print(f"\n--- Results ---")
print(f"PPO avg reward:    {ppo_mean:.2f} +/- {np.std(ppo_rewards):.2f}")
print(f"Random avg reward: {random_mean:.2f} +/- {np.std(random_rewards):.2f}")
print(f"Improvement:       {improvement:.1f}%")

## 5. Visualize Results

Plotting the per-episode returns $G_i$ for both policies. A well-trained agent should show:

$$\hat{J}(\pi_\theta) \gg \hat{J}(\pi_{\text{rand}})$$

The bar chart compares $G_i^{\text{PPO}}$ vs $G_i^{\text{rand}}$ for each evaluation episode. Higher (less negative) rewards indicate lower total waiting time across all vehicles in the network.

With sufficient training ($\geq 100k$ timesteps), PPO typically learns to:
- Minimize average queue length: $\bar{q} = \frac{1}{L} \sum_{i=1}^{L} q_i$
- Reduce cumulative delay: $D = \sum_{t} \sum_{v} w_v^{(t)}$
- Adapt phase durations to match traffic demand asymmetry

In [ ]:
N_EPS = len(ppo_rewards)

# --- Plot 1: Per-episode bar comparison ---
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(N_EPS)
width = 0.35
ax.bar(x - width/2, ppo_rewards, width, label='PPO Agent', color='#2ecc71')
ax.bar(x + width/2, random_rewards, width, label='Random', color='#e74c3c')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title(f'PPO vs Random - {SCENARIO} ({TOTAL_TIMESTEPS:,} training steps)')
ax.set_xticks(x)
ax.set_xticklabels([f'Ep {i+1}' for i in range(N_EPS)])
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# --- Plot 2: Mean performance with error bars ---
fig, ax = plt.subplots(figsize=(7, 5))
labels = ["PPO Agent", "Random Baseline"]
means = [ppo_mean, random_mean]
stds = [float(np.std(ppo_rewards)), float(np.std(random_rewards))]
colors = ["#2ecc71", "#e74c3c"]
ax.bar(labels, means, yerr=stds, capsize=10, color=colors, alpha=0.85,
       edgecolor="black", linewidth=1)
ax.set_ylabel("Average Episode Reward")
ax.set_title(f"Mean Performance Comparison (n={N_EPS} episodes)")
ax.grid(axis="y", alpha=0.3)
for i, m in enumerate(means):
    ax.text(i, m, f"  {m:.2f}", ha="center",
            va="bottom" if m > 0 else "top", fontweight="bold")
plt.tight_layout()
plt.show()

# --- Plot 3: Per-step reward trajectory (Episode 1) ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ppo_step_rewards, label="PPO Agent", color="#2ecc71", linewidth=1.5)
ax.plot(random_step_rewards, label="Random", color="#e74c3c", alpha=0.7, linewidth=1.0)
ax.set_xlabel("Environment Step")
ax.set_ylabel("Per-Step Reward")
ax.set_title("Reward Trajectory During Evaluation (Episode 1)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Plot 4: Cumulative reward over the episode ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.cumsum(ppo_step_rewards), label="PPO Agent", color="#2ecc71", linewidth=2)
ax.plot(np.cumsum(random_step_rewards), label="Random", color="#e74c3c", linewidth=2)
ax.set_xlabel("Environment Step")
ax.set_ylabel("Cumulative Reward")
ax.set_title("Cumulative Reward Over Evaluation Episode")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nPPO outperforms random by {ppo_mean - random_mean:.2f} reward points ({improvement:.1f}% improvement)")

## 6. Save Model (optional)

Saves the learned parameters $\theta$ (policy network) and $\phi$ (value network) to disk. The saved model encodes:

$$\pi_\theta : \mathcal{S} \rightarrow \Delta(\mathcal{A})$$

This mapping from states to action probabilities can be loaded later for deployment or further training via transfer learning on different intersection geometries.

In [ ]:
model.save("ppo_traffic_signal")
print("Model saved to ppo_traffic_signal.zip")

# Download from Colab:
# from google.colab import files
# files.download("ppo_traffic_signal.zip")

## 7. Real-World Evaluation: Toomer's Corner Dataset

Beyond the SUMO synthetic scenario above, this project includes **real traffic data** recorded from Toomer's Corner (Auburn, AL) across 4 days, processed with YOLOv8 vehicle detection. The dataset contains **79 minutes of minute-by-minute vehicle counts (36,883 vehicles total)** across 7 recording sessions.

### Evaluation Setup

We evaluate the three baseline controllers — Fixed-Time, Actuated, and Max-Pressure — against each recorded session. For each session with minute-level counts $c_{d,m}$ per direction $d \in \{N, S, E, W\}$ and minute $m$, we compute per-lane arrival rates:

$$\lambda_{d,m} = \frac{c_{d,m}}{60 \cdot L_d}$$

where $L_d$ is the number of lanes in direction $d$ (Toomer's Corner: $L_N = L_S = 2$, $L_E = L_W = 1$).

### Metrics

For each controller $\pi$ and session $s$ we record:
- **Vehicles passed**: $\text{Pass}(\pi, s) = \sum_{t} \mathbb{1}[\text{vehicle crossed at time } t]$
- **Average waiting time**: $\bar{W}(\pi, s) = \frac{1}{|\text{arrivals}|} \sum_v w_v$
- **Throughput**: $T(\pi, s) = \text{Pass}(\pi, s) / \text{duration}$

### 7.1 Build the Consolidated Dataset

Merge the 7 `Session_*/` folders into a single CSV and compute per-session aggregates.

In [ ]:
!python traffic_rl_project/build_dataset.py

In [ ]:
from IPython.display import Image, display
display(Image("traffic_rl_project/data/dataset_summary.png"))

### 7.2 Run Controller Evaluation Across All Sessions

For each of the 7 sessions and each of the 3 controllers, simulate the intersection using the recorded arrival rates. This produces the per-session metrics table and aggregate plots.

In [ ]:
!python traffic_rl_project/evaluate_dataset.py

### 7.3 Per-Session Results Table

In [ ]:
import pandas as pd
summary_df = pd.read_csv("traffic_rl_project/results/dataset/all_sessions_summary.csv")
summary_df

### 7.4 Aggregate Visualizations

Three plots summarize the cross-session comparison:

1. **Vehicles passed per session** — per-controller bars for every session, showing where each policy wins or fails
2. **Average waiting time per session** — lower is better; reveals stability across varying demand
3. **Aggregate summary** — three panels combining totals, mean throughput, and mean waiting across all 7 sessions

In [ ]:
for fname in [
    "traffic_rl_project/results/dataset/01_passed_by_session.png",
    "traffic_rl_project/results/dataset/02_waiting_by_session.png",
    "traffic_rl_project/results/dataset/03_aggregate_summary.png",
]:
    display(Image(fname))

### 7.5 Interpretation

At Toomer's Corner, South dominates every session with 3–6× the flow of any other approach. Under this heavy asymmetry:

- **Fixed-Time** cycles reliably — no starvation, consistent throughput across all sessions
- **Actuated** underperforms — it keeps switching on "waiting" detections but wastes yellow-phase transitions
- **Max-Pressure** shows a bimodal pattern: it wins big on some sessions (high throughput) but collapses on others when the heaviest queue saturates at capacity, producing the "starvation trap"

This real-world evaluation demonstrates that **controller choice depends on traffic regime**. A policy trained under average conditions may fail under extreme asymmetry — motivation for training adaptive RL agents on diverse scenarios rather than a single representative one.